# SoA-only Legal Proposition (LP) Extraction + Expert Audit (Llama 3.1)

This notebook implements an end-to-end, reproducible workflow:

1. Load **Summary of Argument (SoA)** paragraphs from a **CSV** (one row per paragraph, stable paragraph IDs).
2. Chunk paragraphs (SoA-only; **no ToC/section headings**).
3. Extract **contestable legal propositions (LPs)** with **Llama 3.1 70B Instruct** (4-bit quantized).
4. Export an **expert audit sheet (CSV)** with LP text + evidence paragraph IDs **and paragraph text**.
5. Apply expert decisions (**accept/edit/merge/reject/split**) deterministically (no LLM) to produce a finalized LP list.

**Project root**: `scotus_stance/` (all inputs/outputs live under this folder).


In [3]:
# Cell 0 — Global configuration and robust project paths
# Notes:
# - This cell defines the single source of truth for project paths.
# - All downstream cells MUST derive paths from ROOT_DIR.
# - No relative paths like "scotus_stance/..." are allowed anywhere.

from __future__ import annotations

import os
from pathlib import Path
import random
import numpy as np
import torch

# -----------------
# (A) Reproducibility
# -----------------
SEED: int = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -----------------
# (B) Project root detection
# -----------------
def find_project_root(start: Path | None = None, marker: str = "scotus_stance") -> Path:
    """
    Walk upward from the given path (or CWD) until a directory named `marker` is found.
    This avoids assuming the current working directory.
    """
    here = (start or Path.cwd()).resolve()
    for p in [here] + list(here.parents):
        if p.name == marker:
            return p
    raise FileNotFoundError(f"Could not locate project root '{marker}' from {here}")

ROOT_DIR: Path = find_project_root()

# -----------------
# (C) Canonical project directories (output-only pipeline)
# -----------------
OUTPUT_DIR: Path = ROOT_DIR / "output"
FINAL_DIR: Path = OUTPUT_DIR / "final"
AUDIT_DIR: Path = OUTPUT_DIR / "expert_audited"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# -----------------
# (D) Canonical input / output files
# -----------------
SOA_DIR: Path = OUTPUT_DIR / "soa_paragraphs_v1"
SOA_CSV_PATH: Path = SOA_DIR / "soa_paragraphs.csv"


RAW_JSONL_PATH: Path = OUTPUT_DIR / "lps_raw.jsonl"
AUDIT_SHEET_PATH: Path = OUTPUT_DIR / "lps_audit.csv"

FINAL_JSONL_PATH: Path = FINAL_DIR / "lps_final.jsonl"
FINAL_CSV_PATH: Path = FINAL_DIR / "lps_final.csv"

AUDITED_CSV_PATH: Path = (
    AUDIT_DIR / "lps_audit_simulated_with_merge_and_split.csv"
)

# -----------------
# (E) Chunking parameters
# -----------------
CHUNK_SIZE: int = 8
CHUNK_OVERLAP: int = 2

# -----------------
# (F) Model selection 
# -----------------
MODEL_ID: str = "meta-llama/Llama-3.1-8B-Instruct"
RUN_TAG: str = "soa_only_llama31_8b"

# -----------------
# (G) Hugging Face runtime behavior
# -----------------
# Disable hub progress bars to avoid misleading "0%" stalls in notebook UI.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Use a project-local Hugging Face cache to avoid shared-server lock contention.
os.environ["HF_HOME"] = str(ROOT_DIR / ".hf_cache")

print("ROOT_DIR:", ROOT_DIR)
print("SOA_CSV_PATH:", SOA_CSV_PATH)
print("MODEL_ID:", MODEL_ID)
print("HF_HOME:", os.environ.get("HF_HOME"))


ROOT_DIR: /home/jupyter-hzh308@uky.edu/scotus_stance
SOA_CSV_PATH: /home/jupyter-hzh308@uky.edu/scotus_stance/output/soa_paragraphs_v1/soa_paragraphs.csv
MODEL_ID: meta-llama/Llama-3.1-8B-Instruct
HF_HOME: /home/jupyter-hzh308@uky.edu/scotus_stance/.hf_cache


In [4]:
# Cell 1 — Load SoA paragraphs from CSV and normalize text
# Notes:
# - SoA input is CSV-only by project convention.
# - This cell must NOT guess or search for alternative paths.
# - Encoding normalization is applied once here.

from __future__ import annotations

import pandas as pd
import re

# -----------------
# (A) Load SoA CSV
# -----------------
if not SOA_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Expected SoA CSV not found at: {SOA_CSV_PATH}"
    )

soa_df = pd.read_csv(SOA_CSV_PATH)

required_cols = {"case_group", "brief_uid", "brief_role", "para_id", "para_text"}
missing = required_cols - set(soa_df.columns)
if missing:
    raise ValueError(f"Missing required columns in SoA CSV: {missing}")

# -----------------
# (B) Mojibake / encoding normalization
# -----------------
MOJIBAKE_MAP = {
    "‚Äô": "’",
    "‚Äú": "“",
    "‚Äù": "”",
    "‚Äì": "–",
    "‚Äî": "—",
    "‚Ä¶": "¶",
}

def normalize_text(text: str) -> str:
    """Normalize common mojibake artifacts introduced by PDF extraction."""
    if not isinstance(text, str):
        return text
    for bad, good in MOJIBAKE_MAP.items():
        text = text.replace(bad, good)
    text = re.sub(r"\s+", " ", text).strip()
    return text

soa_df["para_text"] = soa_df["para_text"].apply(normalize_text)

# -----------------
# (C) Basic sanity checks
# -----------------
assert not soa_df.empty, "SoA DataFrame is empty after loading."
assert soa_df["para_text"].notnull().all(), "Null paragraph text detected."
# Each brief_uid should map to exactly one role and one case_group
assert (soa_df.groupby("brief_uid")["brief_role"].nunique() == 1).all(), "brief_uid maps to multiple brief_role values"
assert (soa_df.groupby("brief_uid")["case_group"].nunique() == 1).all(), "brief_uid maps to multiple case_group values"


print("Loaded SoA paragraphs:", len(soa_df))
print("Cases:", soa_df["case_group"].nunique())
print("Roles:", sorted(soa_df["brief_role"].unique()))
print("Briefs:", soa_df["brief_uid"].nunique())

# -----------------
# (D) Export row-wise representation for downstream cells
# -----------------
# Downstream chunking logic operates on row dictionaries
# to avoid tight coupling to pandas APIs.

soa_rows = soa_df.to_dict(orient="records")

print("Exported soa_rows:", len(soa_rows))


Loaded SoA paragraphs: 69
Cases: 4
Roles: ['petitioner', 'respondent']
Briefs: 9
Exported soa_rows: 69


In [3]:
# Cell 2 — SoA-only chunking (no ToC)
# Goal:
# - Chunk paragraphs strictly within a single brief (PDF-level)
# - Guarantee that each LLM invocation sees exactly one brief
# - Preserve brief_role for role labeling, but never use it as a grouping key

from collections import defaultdict

def make_chunks_for_role(rows, chunk_size: int, overlap: int):
    assert chunk_size > 0
    assert 0 <= overlap < chunk_size

    chunks = []

    # Group strictly by brief (not by case+role)
    by_brief = defaultdict(list)
    for r in rows:
        by_brief[(str(r["case_group"]), str(r["brief_uid"]))].append(r)

    for (case_group, brief_uid), items in by_brief.items():
        # Each brief_uid should correspond to exactly one role
        brief_role = str(items[0]["brief_role"])

        # Ensure deterministic paragraph order within a brief
        items = sorted(items, key=lambda x: x["para_id"])

        paras = [
            {"para_id": it["para_id"], "para_text": it["para_text"]}
            for it in items
        ]

        step = chunk_size - overlap
        idx = 0
        c = 1

        while idx < len(paras):
            window = paras[idx: idx + chunk_size]
            if not window:
                break

            # Chunk IDs must be unique across briefs
            chunk_id = f"{brief_uid}_C{c:03d}"

            chunks.append({
                "case_group": case_group,
                "brief_uid": brief_uid,
                "brief_role": brief_role,
                "chunk_id": chunk_id,
                "paragraphs": window,
                "para_ids": [p["para_id"] for p in window],
            })

            idx += step
            c += 1

    return chunks


chunks = make_chunks_for_role(soa_rows, CHUNK_SIZE, CHUNK_OVERLAP)

print("Total chunks:", len(chunks))
if chunks:
    ch0 = chunks[0]
    print(
        "First chunk:",
        ch0["chunk_id"],
        "| case_group:", ch0["case_group"],
        "| brief_uid:", ch0["brief_uid"],
        "| role:", ch0["brief_role"],
        "| paras:", len(ch0["paragraphs"]),
        "| para_ids:", ch0["para_ids"][:5],
    )


Total chunks: 15
First chunk: 23-1122-petitioners_C001 | case_group: 23-1122 | brief_uid: 23-1122-petitioners | role: petitioner | paras: 7 | para_ids: ['p1', 'p2', 'p3', 'p4', 'p5']


In [4]:
# Cell 3 — Load Llama 3.1-8B-Instruct (local, single-GPU, stable)
# -------------------------------------------------------------------
# This project runs in a JupyterHub environment where only a subset of GPUs
# may be exposed to the user session. In the current environment, only one
# RTX 4090 (24GB) is visible.
#
# We therefore use Llama-3.1-8B-Instruct as the primary development model:
# - Fits on a single 24GB GPU in FP16
# - Avoids bitsandbytes / 4-bit quantization instability
# - Loads fully offline from a project-local pinned snapshot
# - Uses explicit pad_token handling for reliable generation

from __future__ import annotations

from pathlib import Path
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

# -------------------------
# (A) Project root (single source of truth)
# -------------------------
# ROOT_DIR is defined in Cell 0 and reused here intentionally
assert "ROOT_DIR" in globals(), "ROOT_DIR must be defined in Cell 0"

# -------------------------
# (B) Local model directory
# -------------------------
MODEL_DIR = ROOT_DIR / "models" / "meta-llama--Llama-3.1-8B-Instruct"
if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Local model directory not found: {MODEL_DIR}")

# Ensure we do not hit Hugging Face Hub during model load
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")

# -------------------------
# (C) Load tokenizer (offline)
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    use_fast=True,
    local_files_only=True,
)

# Explicitly define pad token to avoid generation warnings
# LLaMA-style models do not define a pad token by default
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# -------------------------
# (D) Load model (offline, FP16, single GPU)
# -------------------------
model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR),
    device_map="auto",          # resolves to a single visible GPU in JupyterHub
    torch_dtype=torch.float16,  # optimal for RTX 4090
    local_files_only=True,
)

model.eval()

# -------------------------
# (E) Generation defaults (deterministic)
# -------------------------
GENCFG = GenerationConfig(
    do_sample=False,
    repetition_penalty=1.05,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    use_cache=True,  # may be overridden in Cell 5 for memory-stability debugging
)

# -------------------------
# (F) Availability / sanity checks
# -------------------------
print("Loaded model (local):", MODEL_DIR.name)
print("CUDA available:", torch.cuda.is_available())
print("CUDA devices visible:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Model device:", model.device)
    try:
        free, total = torch.cuda.mem_get_info()
        print(f"CUDA memory: free={free/1e9:.1f}GB total={total/1e9:.1f}GB")
    except Exception:
        pass


/home/jupyter-hzh308@uky.edu/venvs/lp14b/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


Loaded model (local): meta-llama--Llama-3.1-8B-Instruct
CUDA available: True
CUDA devices visible: 1
Model device: cuda:0
CUDA memory: free=7.5GB total=25.3GB


In [5]:
# Cell 4 — SoA-only LP extraction prompt (expert-revised, JSON-only)
#
# This version:
# - Uses the expert-revised legal prompt verbatim (minus ToC/bundle references)
# - Operates ONLY on Summary of Argument paragraphs
# - Preserves party framing and contestability
# - Returns JSON only (for downstream audit + CSV export)

from __future__ import annotations

import json
from typing import Any, Dict, List

# -------------------------------------------------------------------
# Prompt versioning (required by downstream logging)
# -------------------------------------------------------------------
PROMPT_VERSION = "soa_only_expert_v1_json_only"

# -------------------------------------------------------------------
# Robust JSON extraction helper (brace balancing)
# -------------------------------------------------------------------
def extract_first_json_object(text: str) -> Dict[str, Any]:
    """Extract the first complete JSON object using brace balancing."""
    t = (text or "").strip()

    # Fast path: if the whole output is valid JSON dict
    try:
        obj = json.loads(t)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    # Start scanning from the first '{' to avoid leading junk
    first_brace = t.find("{")
    if first_brace == -1:
        raise ValueError("No '{' found in text; cannot extract JSON object.")
    t = t[first_brace:]

    in_str = False
    esc = False
    depth = 0
    start = None

    for i, ch in enumerate(t):
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue

        # Not in a string
        if ch == '"':
            in_str = True
            continue

        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            if depth > 0:
                depth -= 1
                if depth == 0 and start is not None:
                    blob = t[start : i + 1]
                    try:
                        obj = json.loads(blob)
                        if isinstance(obj, dict):
                            return obj
                    except Exception:
                        start = None
                        continue

    raise ValueError("Unbalanced braces: could not extract a full JSON object.")

# -----------------
# (A) System prompt (expert-revised)
# -----------------
SYSTEM_PROMPT = (
    "You are a legal analyst focused on extracting contestable legal propositions "
    "that a judge could reasonably support, oppose, or probe during oral argument. "
    "Your goal is NOT to restate ultimate legal conclusions or holdings. "
    "Instead, extract arguable issue-framing propositions that preserve party framing "
    "and leave room for disagreement. "
    "Return JSON only. No prose."
)

# -----------------
# (B) User prompt builder (SoA-only; ToC removed)
# -----------------
def build_user_prompt(chunk: Dict[str, Any]) -> str:
    """
    Build a SoA-only user prompt based on expert-revised instructions.
    All references to ToC, issue bundles, and subheadings are removed.
    """
    
    # --- REQUIRED chunk schema (brief-level guarantee) ---
    for k in ("case_group", "brief_uid", "brief_role", "chunk_id", "paragraphs"):
        assert k in chunk, f"Missing key in chunk: {k}"

    
    # NOTE: chunk structure is produced by Cell 2:
    # chunk["paragraphs"] is a list of {"para_id": ..., "para_text": ...}
    paras_text = "\n\n".join(
        [f"[{p['para_id']}]\n{p['para_text']}" for p in chunk["paragraphs"]]
    )

    # Explicit schema reminder to reduce non-JSON / malformed outputs
    schema_hint = (
        "{\n"
        '  "lps": [\n'
        "    {\n"
        '      "lp_text": "string",\n'
        '      "evidence_para_ids": ["p1", "p3"]\n'
        "    }\n"
        "  ]\n"
        "}"
    )

    return (
    f"Case: {chunk['case_group']}\n"
    f"Brief uid: {chunk['brief_uid']}\n"
    f"Brief role: {chunk['brief_role']}\n"
    f"Chunk id: {chunk['chunk_id']}\n\n"
    "Summary of Argument paragraphs (each has an id):\n"
    f"{paras_text}\n\n"
    "Task:\n"
    "- Extract legal propositions (LPs) substantively advanced here.\n"
    "- LPs must be case-grounded and may preserve party framing; do NOT neutralize; do NOT over-claim.\n"
    "- Prefer LPs that are contestable: a reasonable judge could challenge, question, or partially disagree.\n"
    "- Avoid 'ultimate conclusion' LPs that are too final or too dispositive "
    "(e.g., 'the statute is constitutional', "
    "'the law satisfies any level of scrutiny', "
    "'strict scrutiny is met').\n"
    "- Avoid policy statements or value statements that discuss the consequences "
    "if a particular proposition or position were adopted.\n"
    "- Avoid universally agreed principles "
    "(e.g., 'the First Amendment protects freedom of speech', "
    "'the government should not pick ideological winners and losers').\n"
    "- When paragraphs contain multiple contestable determinations, prefer SPLITTING into multiple LPs.\n"
    "  * If a sentence contains more than one inferential step, each step shall become its own proposition "
    "(trigger words include: because, unless, except, therefore, etc.).\n"
    "  * Split when a judge could agree with part of the LP and disagree with another part.\n"
    "- Ignore purely citation-based propositions that only recite other cases without applying them.\n"
    "- Merge near-duplicates ONLY when they express the same contestable judgment "
    "(not merely the same bottom-line conclusion).\n\n"
    "LP definition:\n"
    "- An LP is a normative legal claim advanced by a party that frames an issue "
    "in a way that invites judicial evaluation.\n"
    "- It should read like a claim that could be pressed or resisted in oral argument.\n\n"
    "Output requirements:\n"
    "- Return a single JSON object with a key 'lps'.\n"
    "- 'lps' must be a list of objects; each object has: lp_text, evidence_para_ids.\n"
    "- evidence_para_ids must include ALL paragraph ids that substantively advance that LP.\n"
    "- Return JSON only (no markdown, no commentary, no extra keys).\n"
    "- Follow this exact schema:\n"
    f"{schema_hint}\n\n"
    "Additional guidance:\n"
    "- If the paragraphs discuss whether the statute targets a viewpoint or applies regardless of message, "
    "extract a separate LP explicitly framed in terms of viewpoint neutrality.\n"
    "- If the argument discusses whether selling a service communicates approval, endorsement, or signaling "
    "of the buyer, express this explicitly (e.g., approval or endorsement), not only in doctrinal terms.\n"
    "- If resolution is argued to depend on the nature or content of future products, "
    "hypothetical applications, or the need for a fuller factual record, "
    "extract a separate LP concerning ripeness or lack of an adequate record."
)


# -----------------
# (C) Minimal deterministic preview
# -----------------
assert "chunks" in globals() and chunks, "chunks not found; run Cell 2."
print("Prompt version:", PROMPT_VERSION)
print("Preview (first 400 chars):")
print(build_user_prompt(chunks[0])[:400])


Prompt version: soa_only_expert_v1_json_only
Preview (first 400 chars):
Case: 23-1122
Brief uid: 23-1122-petitioners
Brief role: petitioner
Chunk id: 23-1122-petitioners_C001

Summary of Argument paragraphs (each has an id):
[p1]
H.B. 1181’s age-verification requirement is unconstitutional under a straightforward application of First Amendment principles and precedent. The preliminary injunction should accordingly be restored.

[p2]
A. When speech is protected by the 


In [7]:
# Cell 5 — Run LLM extraction over chunks and write raw JSONL (one LP per line)
# -------------------------------------------------------------------
# Stability-first observability upgrades:
# - Uses MAX_NEW_TOKENS=768 by default (more stable JSON closure, slower)
# - Retries once on likely truncation/unbalanced braces with a larger budget (RETRY_MAX_NEW_TOKENS=1024)
# - Adds generation time limit (GEN_MAX_TIME_SEC) to avoid indefinite hangs
# - Dumps raw generations to output/raw_generations/ for forensic debugging
# - Stage-level timings: prompt -> tokenize -> generate -> decode -> parse -> postprocess
# - GPU memory report (if CUDA available)
# - Incremental JSONL flushing
#
# Brief-level provenance upgrades (this patch):
# - Reads brief_uid from chunks (Cell 2)
# - Logs brief_uid at START
# - Includes brief_uid in parse_fail.json
# - Names raw dump files with brief_uid to avoid collisions
# - Includes brief_uid in row_id hashing and JSONL rows
#
# NEW (this patch): raw_lp_id
# - Adds a deterministic LP identifier (raw_lp_id) for stable expert audit joins
# - raw_lp_id is derived from the same stable source string as row_id, but uses a longer hash

from __future__ import annotations

import time
import json
from copy import deepcopy
from hashlib import sha1
from pathlib import Path
from typing import Any, Dict, List

import torch

# -------------------------
# (A) Project paths (robust to CWD)
# -------------------------
def find_project_root(start: Path | None = None, marker: str = "scotus_stance") -> Path:
    here = (start or Path.cwd()).resolve()
    for p in [here] + list(here.parents):
        if p.name == marker:
            return p
    raise FileNotFoundError(f"Could not locate project root '{marker}' starting from: {here}")

ROOT_DIR = find_project_root()
OUT_DIR = ROOT_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_JSONL = OUT_DIR / "lps_raw.jsonl"
PARSE_FAIL_JSON = OUT_DIR / "parse_fail.json"

RAW_DUMP_DIR = OUT_DIR / "raw_generations"
RAW_DUMP_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# (B) Preconditions
# -------------------------
assert "chunks" in globals() and chunks, "chunks not found; run Cell 2."
assert "SYSTEM_PROMPT" in globals() and "build_user_prompt" in globals(), "prompts not found; run Cell 4."
assert "extract_first_json_object" in globals(), "JSON extractor not found; run Cell 4."
assert "tokenizer" in globals() and "model" in globals(), "model/tokenizer not found; run Cell 3."
assert "GENCFG" in globals(), "GENCFG not found; run Cell 3."
assert "PROMPT_VERSION" in globals(), "PROMPT_VERSION not found; run Cell 4."

# -------------------------
# (C) Run settings (stability-first)
# -------------------------
MAX_NEW_TOKENS = 768               # more stable JSON completion; slower than 256/512
RETRY_ON_PARSE_FAIL = True
RETRY_MAX_NEW_TOKENS = 1024        # retry budget for likely truncation/unbalanced JSON

GEN_MAX_TIME_SEC = 180             # hard wallclock time limit for generate() per chunk
LIMIT_CHUNKS = None                # set to int for debugging (e.g., 4), or None for all
FLUSH_EVERY = 2                    # flush frequently for safety/visibility

# If a stage takes longer than this, print a notice (does not interrupt)
SLOW_STAGE_WARN_SEC = 30

MODEL_TAG = getattr(model.config, "_name_or_path", "local_model")
MODEL_TAG = str(MODEL_TAG)

def _gpu_mem_str() -> str:
    """Return a short GPU memory string if CUDA is available."""
    if not torch.cuda.is_available():
        return "CUDA: not available"
    try:
        free, total = torch.cuda.mem_get_info()
        return (
            f"CUDA mem free={free/1e9:.1f}GB total={total/1e9:.1f}GB "
            f"allocated={torch.cuda.memory_allocated()/1e9:.1f}GB "
            f"reserved={torch.cuda.memory_reserved()/1e9:.1f}GB"
        )
    except Exception:
        return "CUDA mem: unknown"

def _dump_text(path: Path, text: str) -> None:
    """Write raw model output for forensic debugging."""
    try:
        path.write_text(text or "", encoding="utf-8")
    except Exception:
        # Avoid breaking the pipeline on dump failure
        pass

# -------------------------
# (D) Generation helper with stage prints
# -------------------------
@torch.inference_mode()
def llm_generate(
    system_prompt: str,
    user_prompt: str,
    max_new_tokens: int,
    chunk_label: str,
    gen_max_time_sec: int,
) -> Dict[str, Any]:
    """
    Generate completion text for one prompt, with detailed stage timing.
    Returns: {input_tokens, tokenize_seconds, gen_seconds, decode_seconds, output_text, total_seconds}
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    t0 = time.time()

    # Tokenize via chat template
    print(f"[{chunk_label}] TOKENIZE start...")
    enc_t0 = time.time()

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    tokenize_seconds = time.time() - enc_t0
    input_tokens = int(input_ids.shape[1])
    print(f"[{chunk_label}] TOKENIZE done in {tokenize_seconds:.2f}s | input_tokens={input_tokens}")

    # Attention mask: all ones (reliable behavior)
    attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=model.device)

    gen_cfg = deepcopy(GENCFG)
    gen_cfg.max_new_tokens = int(max_new_tokens)

    # Time limit (Transformers supports max_time in seconds)
    # NOTE: max_time is best-effort; it may return early.
    print(f"[{chunk_label}] GENERATE start... | {_gpu_mem_str()} | max_new_tokens={max_new_tokens} max_time={gen_max_time_sec}s")
    gen_t0 = time.time()

    out = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        generation_config=gen_cfg,
        max_time=float(gen_max_time_sec),
    )

    gen_seconds = time.time() - gen_t0
    if gen_seconds > SLOW_STAGE_WARN_SEC:
        print(f"[{chunk_label}] NOTICE: GENERATE took {gen_seconds:.1f}s (>= {SLOW_STAGE_WARN_SEC}s)")

    print(f"[{chunk_label}] GENERATE done in {gen_seconds:.2f}s | {_gpu_mem_str()}")

    # Decode
    print(f"[{chunk_label}] DECODE start...")
    dec_t0 = time.time()

    completion_ids = out[0][input_ids.shape[1]:]
    output_text = tokenizer.decode(completion_ids, skip_special_tokens=True).strip()

    decode_seconds = time.time() - dec_t0
    print(f"[{chunk_label}] DECODE done in {decode_seconds:.2f}s | output_chars={len(output_text)}")

    return {
        "input_tokens": input_tokens,
        "tokenize_seconds": tokenize_seconds,
        "gen_seconds": gen_seconds,
        "decode_seconds": decode_seconds,
        "output_text": output_text,
        "total_seconds": time.time() - t0,
    }

# -------------------------
# (E) Incremental writer
# -------------------------
def append_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    if not rows:
        return
    with path.open("a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# Start fresh each run
RAW_JSONL.write_text("", encoding="utf-8")

rows_out: List[Dict[str, Any]] = []
parse_fail: List[Dict[str, Any]] = []
buffer: List[Dict[str, Any]] = []

total_chunks = len(chunks) if LIMIT_CHUNKS is None else min(len(chunks), int(LIMIT_CHUNKS))

print(f"Running LP extraction on {total_chunks} chunks using expert-revised prompt...")
print("Output JSONL:", RAW_JSONL)
print("Model tag:", MODEL_TAG, "| Prompt version:", PROMPT_VERSION)
print("MAX_NEW_TOKENS:", MAX_NEW_TOKENS, "| RETRY_MAX_NEW_TOKENS:", RETRY_MAX_NEW_TOKENS)
print("FLUSH_EVERY:", FLUSH_EVERY, "| GEN_MAX_TIME_SEC:", GEN_MAX_TIME_SEC)
print("GPU status:", _gpu_mem_str())

t_all = time.time()

# -------------------------
# (F) Main loop
# -------------------------
for i, ch in enumerate(chunks[:total_chunks], start=1):
    case_group = str(ch.get("case_group", ""))
    brief_uid  = str(ch.get("brief_uid", ""))     # NEW: brief-level provenance
    brief_role = str(ch.get("brief_role", ""))
    chunk_id   = str(ch.get("chunk_id", f"chunk_{i:03d}"))
    n_paras = len(ch.get("paragraphs", []))
    label = f"{i}/{total_chunks}:{chunk_id}"

    print(f"\n[{label}] START case={case_group} brief_uid={brief_uid} role={brief_role} paras={n_paras}")

    # Build prompt
    t_prompt = time.time()
    user_prompt = build_user_prompt(ch)
    prompt_seconds = time.time() - t_prompt
    print(f"[{label}] PROMPT built in {prompt_seconds:.2f}s | prompt_chars={len(user_prompt)}")

    # Generate (first attempt)
    try:
        gen_info = llm_generate(
            SYSTEM_PROMPT,
            user_prompt,
            MAX_NEW_TOKENS,
            chunk_label=label,
            gen_max_time_sec=GEN_MAX_TIME_SEC,
        )
        raw = gen_info["output_text"]
        print(
            f"[{label}] STATS tokenize={gen_info['tokenize_seconds']:.2f}s "
            f"generate={gen_info['gen_seconds']:.2f}s decode={gen_info['decode_seconds']:.2f}s "
            f"total={gen_info['total_seconds']:.2f}s"
        )
    except Exception as e:
        parse_fail.append(
            {
                "stage": "generate_or_tokenize",
                "case_group": case_group,
                "brief_uid": brief_uid,      # NEW
                "brief_role": brief_role,
                "chunk_id": chunk_id,
                "error": repr(e),
            }
        )
        print(f"[{label}] ERROR at generate/tokenize: {repr(e)}")
        continue

    # Dump raw generation (attempt 1) — include brief_uid to avoid collisions
    safe_brief_uid = brief_uid if brief_uid else "unknown_brief"
    dump_path = RAW_DUMP_DIR / f"{safe_brief_uid}__{chunk_id}.txt"
    _dump_text(dump_path, raw)

    # Parse JSON (attempt 1)
    t_parse = time.time()
    parsed_obj = None
    lps = None
    parse_error = None

    try:
        parsed_obj = extract_first_json_object(raw)
        lps = parsed_obj.get("lps", [])
        if not isinstance(lps, list):
            raise ValueError("JSON key 'lps' is not a list")
        print(f"[{label}] PARSE ok in {time.time()-t_parse:.2f}s | lps={len(lps)} | dumped -> {dump_path.name}")
    except Exception as e:
        parse_error = e
        print(f"[{label}] PARSE_FAIL ({type(e).__name__}): {repr(e)} | dumped -> {dump_path.name}")

    # Retry on parse fail (attempt 2)
    if (parse_error is not None) and RETRY_ON_PARSE_FAIL:
        likely_trunc = ("Unbalanced braces" in repr(parse_error)) or ("JSONDecodeError" in repr(parse_error))
        if likely_trunc:
            retry_label = f"{label}:RETRY"
            print(f"[{label}] RETRY enabled (likely truncation). Retrying with max_new_tokens={RETRY_MAX_NEW_TOKENS} ...")

            try:
                gen_info2 = llm_generate(
                    SYSTEM_PROMPT,
                    user_prompt,
                    RETRY_MAX_NEW_TOKENS,
                    chunk_label=retry_label,
                    gen_max_time_sec=GEN_MAX_TIME_SEC,
                )
                raw2 = gen_info2["output_text"]
                dump_path2 = RAW_DUMP_DIR / f"{safe_brief_uid}__{chunk_id}.retry.txt"
                _dump_text(dump_path2, raw2)

                t_parse2 = time.time()
                parsed_obj = extract_first_json_object(raw2)
                lps = parsed_obj.get("lps", [])
                if not isinstance(lps, list):
                    raise ValueError("JSON key 'lps' is not a list")

                print(f"[{label}] PARSE ok after retry in {time.time()-t_parse2:.2f}s | lps={len(lps)} | dumped -> {dump_path2.name}")
                parse_error = None
            except Exception as e2:
                parse_fail.append(
                    {
                        "stage": "parse_json_after_retry",
                        "case_group": case_group,
                        "brief_uid": brief_uid,   # NEW
                        "brief_role": brief_role,
                        "chunk_id": chunk_id,
                        "error": repr(e2),
                        "raw_dump": str((RAW_DUMP_DIR / f"{safe_brief_uid}__{chunk_id}.retry.txt").resolve()),
                    }
                )
                print(f"[{label}] PARSE_FAIL after retry: {repr(e2)}")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue
        else:
            parse_fail.append(
                {
                    "stage": "parse_json_no_retry",
                    "case_group": case_group,
                    "brief_uid": brief_uid,      # NEW
                    "brief_role": brief_role,
                    "chunk_id": chunk_id,
                    "error": repr(parse_error),
                    "raw_dump": str(dump_path.resolve()),
                }
            )
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            continue
    elif parse_error is not None:
        parse_fail.append(
            {
                "stage": "parse_json",
                "case_group": case_group,
                "brief_uid": brief_uid,          # NEW
                "brief_role": brief_role,
                "chunk_id": chunk_id,
                "error": repr(parse_error),
                "raw_dump": str(dump_path.resolve()),
            }
        )
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue

    # Postprocess LPs (one LP per row)
    t_post = time.time()
    kept = 0
    dropped = 0

    assert isinstance(lps, list)

    for lp in lps:
        if not isinstance(lp, dict):
            dropped += 1
            continue

        lp_text = (lp.get("lp_text") or "").strip()
        evid = lp.get("evidence_para_ids") or []

        if not lp_text or not isinstance(evid, list) or not evid:
            dropped += 1
            continue

        evid = [str(x).strip() for x in evid if str(x).strip()]
        if not evid:
            dropped += 1
            continue

        # Include brief_uid to prevent collisions across briefs within the same case_group/role
        rid_src = (
            f"{case_group}|{brief_uid}|{brief_role}|{chunk_id}|"
            f"{lp_text}|{';'.join(evid)}|{MODEL_TAG}|{PROMPT_VERSION}"
        )
        row_id = sha1(rid_src.encode("utf-8")).hexdigest()[:12]

        # NEW: deterministic LP id (longer hash for audit joins)
        raw_lp_id = "rawlp_" + sha1(rid_src.encode("utf-8")).hexdigest()[:16]

        row = {
            "row_id": row_id,
            "raw_lp_id": raw_lp_id,            # NEW
            "case_group": case_group,
            "brief_uid": brief_uid,            # NEW
            "brief_role": brief_role,
            "chunk_id": chunk_id,
            "model_id": MODEL_TAG,
            "prompt_version": PROMPT_VERSION,
            "lp_text": lp_text,
            "evidence_para_ids": evid,
        }

        rows_out.append(row)
        buffer.append(row)
        kept += 1

    post_seconds = time.time() - t_post
    print(f"[{label}] END kept={kept} dropped={dropped} postprocess={post_seconds:.2f}s | buffer_rows={len(buffer)} total_rows={len(rows_out)}")

    # Flush incremental JSONL
    if (i % FLUSH_EVERY) == 0 and buffer:
        append_jsonl(RAW_JSONL, buffer)
        print(f"[{label}] FLUSH appended {len(buffer)} rows -> {RAW_JSONL.name}")
        buffer = []

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Final flush
if buffer:
    append_jsonl(RAW_JSONL, buffer)
    print(f"[FINAL] FLUSH appended {len(buffer)} rows -> {RAW_JSONL.name}")

elapsed = time.time() - t_all
print("\nDone.")
print(f"Total LP rows: {len(rows_out)} | Elapsed: {elapsed:.1f}s")
print("Wrote:", RAW_JSONL)

if parse_fail:
    PARSE_FAIL_JSON.write_text(json.dumps(parse_fail, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved parse_fail details ->", PARSE_FAIL_JSON)
else:
    print("No parse failures.")


Running LP extraction on 15 chunks using expert-revised prompt...
Output JSONL: /home/jupyter-hzh308@uky.edu/scotus_stance/output/lps_raw.jsonl
Model tag: /home/jupyter-hzh308@uky.edu/scotus_stance/models/meta-llama--Llama-3.1-8B-Instruct | Prompt version: soa_only_expert_v1_json_only
MAX_NEW_TOKENS: 768 | RETRY_MAX_NEW_TOKENS: 1024
FLUSH_EVERY: 2 | GEN_MAX_TIME_SEC: 180
GPU status: CUDA mem free=7.5GB total=25.3GB allocated=16.1GB reserved=17.2GB

[1/15:23-1122-petitioners_C001] START case=23-1122 brief_uid=23-1122-petitioners role=petitioner paras=7
[1/15:23-1122-petitioners_C001] PROMPT built in 0.00s | prompt_chars=6772
[1/15:23-1122-petitioners_C001] TOKENIZE start...
[1/15:23-1122-petitioners_C001] TOKENIZE done in 0.01s | input_tokens=1484
[1/15:23-1122-petitioners_C001] GENERATE start... | CUDA mem free=7.5GB total=25.3GB allocated=16.1GB reserved=17.2GB | max_new_tokens=768 max_time=180s
[1/15:23-1122-petitioners_C001] GENERATE done in 13.65s | CUDA mem free=7.5GB total=25.3GB

In [8]:
# Cell 6 — Build expert audit CSV (one row per LP; include evidence IDs + evidence text)
# -------------------------------------------------------------------
# This cell:
# - Reads output/lps_raw.jsonl produced in Cell 5
# - Builds an evidence text map from SoA rows (CSV is the authoritative source)
# - Creates an expert audit sheet with empty decision columns
# - Writes output/lps_audit.csv
#
# Brief-level provenance upgrades (this patch):
# - Evidence lookup keys use (case_group, brief_uid, para_id) instead of (case_group, brief_role, para_id)
# - Requires brief_uid in lps_raw.jsonl and includes it in the audit sheet
#
# NEW (this patch): raw_lp_id
# - Requires raw_lp_id in lps_raw.jsonl and includes it in the audit sheet for stable expert joins

from __future__ import annotations

import pandas as pd
from pathlib import Path
from typing import Any, Dict, List, Tuple

# -------------------------
# (A) Project paths (robust to CWD)
# -------------------------
def find_project_root(start: Path | None = None, marker: str = "scotus_stance") -> Path:
    here = (start or Path.cwd()).resolve()
    for p in [here] + list(here.parents):
        if p.name == marker:
            return p
    raise FileNotFoundError(f"Could not locate project root '{marker}' starting from: {here}")

ROOT_DIR = find_project_root()
OUT_DIR = ROOT_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_JSONL = OUT_DIR / "lps_raw.jsonl"
AUDIT_CSV = OUT_DIR / "lps_audit.csv"

# -------------------------
# (B) Preconditions
# -------------------------
assert RAW_JSONL.exists(), f"Raw JSONL not found: {RAW_JSONL}"
assert "soa_rows" in globals() and isinstance(soa_rows, list) and soa_rows, "soa_rows not found; run Cell 1."

# -------------------------
# (C) Build para_text_map from SoA rows (authoritative)
# -------------------------
# Key MUST be brief-level to avoid cross-brief contamination:
# (case_group, brief_uid, para_id) is stable even if a case has multiple briefs per role.
para_text_map: Dict[Tuple[str, str, str], str] = {}
for r in soa_rows:
    key = (str(r["case_group"]), str(r["brief_uid"]), str(r["para_id"]))
    para_text_map[key] = str(r.get("para_text", ""))

def join_evidence_text(case_group: str, brief_uid: str, para_ids: List[str]) -> str:
    """Join evidence paragraph text in a stable, readable format."""
    parts: List[str] = []
    for pid in (para_ids or []):
        key = (str(case_group), str(brief_uid), str(pid))
        txt = para_text_map.get(key, "")
        parts.append(f"[{pid}] {txt}")
    return "\n\n".join(parts)

# -------------------------
# (D) Load LP rows and add audit columns
# -------------------------
df_lps = pd.read_json(RAW_JSONL, lines=True)

required_cols = {
    "row_id", "raw_lp_id", "case_group", "brief_uid", "brief_role", "chunk_id",
    "model_id", "prompt_version",
    "lp_text", "evidence_para_ids",
}
missing = required_cols - set(df_lps.columns)
if missing:
    raise ValueError(f"Missing required columns in lps_raw.jsonl: {missing}")

df_lps["evidence_para_ids_joined"] = df_lps["evidence_para_ids"].map(
    lambda xs: ";".join([str(x) for x in xs]) if isinstance(xs, list) else ""
)

df_lps["evidence_text"] = df_lps.apply(
    lambda r: join_evidence_text(r["case_group"], r["brief_uid"], r["evidence_para_ids"]),
    axis=1,
)

# Expert columns (empty by default)
df_lps["decision"] = ""            # e.g., accept/edit/merge/split
df_lps["edit_lp_text"] = ""        # revised LP text if edited
df_lps["merge_group_id"] = ""      # group identifier if merged
df_lps["split_children"] = ""      # use ' ||| ' delimiter for split results
df_lps["notes"] = ""               # optional notes

cols = [
    "row_id", "raw_lp_id", "case_group", "brief_uid", "brief_role", "chunk_id",
    "model_id", "prompt_version",
    "lp_text", "evidence_para_ids_joined", "evidence_text",
    "decision", "edit_lp_text", "merge_group_id", "split_children", "notes",
]

df_audit = df_lps[cols].copy()
df_audit.to_csv(AUDIT_CSV, index=False, encoding="utf-8-sig")

print("Wrote audit sheet:", AUDIT_CSV)
print("Rows:", len(df_audit))
print("Cases:", df_audit["case_group"].nunique())
print("Briefs:", df_audit["brief_uid"].nunique())
print("Roles:", sorted(df_audit["brief_role"].unique()))


Wrote audit sheet: /home/jupyter-hzh308@uky.edu/scotus_stance/output/lps_audit.csv
Rows: 174
Cases: 4
Briefs: 9
Roles: ['petitioner', 'respondent']


In [5]:
# Cell 7 — Apply expert-audited CSV deterministically to produce final LP list (no LLM)
# -------------------------------------------------------------------
# This cell:
# - Reads an expert-audited CSV produced by human experts (HITL)
#   NOTE: The CSV is expected at:
#         input/expert_audited/expert_audited_lps.csv
# - Applies accept / edit / merge / split deterministically (no model calls)
# - Produces finalized LPs for stance targets
# - Writes output/final/lps_final.jsonl and output/final/lps_final.csv
#
# Brief-level provenance upgrades (this patch):
# - Evidence lookup keys use (case_group, brief_uid, para_id) instead of (case_group, brief_role, para_id)
# - Requires brief_uid in audited CSV
# - Prevents merges from accidentally crossing briefs by grouping merges within (case_group, brief_uid, brief_role, merge_group_id)
# - Carries brief_uid into final outputs and final_lp_id hashing sources
#
# NEW (this patch): raw_lp_id passthrough
# - Requires raw_lp_id in audited CSV
# - For accept/edit (and other non-merge, non-split decisions), set final_lp_id = raw_lp_id

from __future__ import annotations

import json
import re
from hashlib import sha1
from pathlib import Path
from typing import Any, Dict, List, Tuple

import pandas as pd

# -------------------------
# (A) Project paths (robust to CWD)
# -------------------------
def find_project_root(start: Path | None = None, marker: str = "scotus_stance") -> Path:
    here = (start or Path.cwd()).resolve()
    for p in [here] + list(here.parents):
        if p.name == marker:
            return p
    raise FileNotFoundError(f"Could not locate project root '{marker}' starting from: {here}")

ROOT_DIR = find_project_root()

# Expert-audited input (authoritative HITL output)
AUDITED_PATH = (
    ROOT_DIR
    / "input"
    / "expert_audited"
    / "expert_audited_lps.csv"
)

# Final outputs
OUT_DIR = ROOT_DIR / "output"
FINAL_DIR = OUT_DIR / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_JSONL = FINAL_DIR / "lps_final.jsonl"
FINAL_CSV = FINAL_DIR / "lps_final.csv"

# -------------------------
# (B) Preconditions
# -------------------------
assert AUDITED_PATH.exists(), f"Audited CSV not found: {AUDITED_PATH}"
assert "soa_rows" in globals() and isinstance(soa_rows, list) and soa_rows, \
    "soa_rows not found; run Cell 1."

print("Using expert-audited CSV:", AUDITED_PATH)

# -------------------------
# (C) Evidence lookup from SoA (authoritative)
# -------------------------
# Key MUST be brief-level to avoid cross-brief contamination:
# (case_group, brief_uid, para_id)
para_text_map: Dict[Tuple[str, str, str], str] = {}
for r in soa_rows:
    key = (str(r["case_group"]), str(r["brief_uid"]), str(r["para_id"]))
    para_text_map[key] = str(r.get("para_text", ""))

def parse_evidence_ids(s: str) -> List[str]:
    s = (s or "").strip()
    if not s:
        return []
    return [x.strip() for x in s.split(";") if x.strip()]

def sort_pid(pid: str):
    # Prefer pN numeric sorting when possible
    if re.match(r"^p\d+$", pid):
        return (0, int(pid[1:]))
    return (1, pid)

def join_evidence_text(case_group: str, brief_uid: str, para_ids: List[str]) -> str:
    parts: List[str] = []
    for pid in para_ids:
        key = (str(case_group), str(brief_uid), str(pid))
        txt = para_text_map.get(key, "")
        parts.append(f"[{pid}] {txt}")
    return "\n\n".join(parts)

# -------------------------
# (D) Load audited decisions and normalize columns
# -------------------------
df = pd.read_csv(AUDITED_PATH)

required_cols = {
    "row_id",
    "raw_lp_id",                # NEW: required for stable final_lp_id passthrough
    "case_group",
    "brief_uid",                 # NEW: required for brief-level provenance
    "brief_role",
    "lp_text",
    "decision",
    "edit_lp_text",
    "merge_group_id",
    "split_children",
    "evidence_para_ids_joined",
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in audited CSV: {missing}")

df["decision"] = df["decision"].fillna("").astype(str).str.strip().str.lower()
df["edit_lp_text"] = df["edit_lp_text"].fillna("").astype(str)
df["merge_group_id"] = df["merge_group_id"].fillna("").astype(str).str.strip()
df["split_children"] = df["split_children"].fillna("").astype(str)
df["evidence_para_ids_joined"] = df["evidence_para_ids_joined"].fillna("").astype(str)

# NEW: normalize raw_lp_id
df["raw_lp_id"] = df["raw_lp_id"].fillna("").astype(str).str.strip()

print("Decision counts:", df["decision"].value_counts(dropna=False).to_dict())

# -------------------------
# (E) Apply merge groups first (must NOT cross briefs)
# -------------------------
final_rows: List[Dict[str, Any]] = []
merged_done = set()

merge_df = df[df["decision"] == "merge"].copy()
# Only meaningful merge rows have a non-empty merge_group_id
merge_df = merge_df[merge_df["merge_group_id"].astype(str).str.strip() != ""]

# Group merges within a brief boundary to avoid cross-brief merges:
# (case_group, brief_uid, brief_role, merge_group_id)
for (case_group, brief_uid, brief_role, gid), grp in merge_df.groupby(
    ["case_group", "brief_uid", "brief_role", "merge_group_id"],
    dropna=False,
):
    case_group = str(case_group)
    brief_uid = str(brief_uid)
    brief_role = str(brief_role)
    gid = str(gid).strip()
    if not gid:
        continue

    row_ids = grp["row_id"].astype(str).tolist()

    # Choose merged text deterministically:
    # 1) longest non-empty edit_lp_text
    # 2) otherwise longest lp_text
    candidates: List[str] = []
    for _, r in grp.iterrows():
        t = str(r.get("edit_lp_text", "")).strip()
        if t:
            candidates.append(t)
    if not candidates:
        for _, r in grp.iterrows():
            t = str(r.get("lp_text", "")).strip()
            if t:
                candidates.append(t)
    merged_text = max(candidates, key=len) if candidates else ""

    # Union evidence ids
    evid: List[str] = []
    for s in grp["evidence_para_ids_joined"].tolist():
        evid.extend(parse_evidence_ids(s))
    evid = sorted(set(evid), key=sort_pid)

    evidence_text = join_evidence_text(case_group, brief_uid, evid)

    rid_src = (
        f"{case_group}|{brief_uid}|{brief_role}|MERGE|{gid}|"
        f"{merged_text}|{';'.join(evid)}|{';'.join(row_ids)}"
    )
    fid = "F-" + sha1(rid_src.encode("utf-8")).hexdigest()[:10]

    final_rows.append(
        {
            "case_group": case_group,
            "brief_uid": brief_uid,
            "brief_role": brief_role,
            "final_lp_id": fid,
            "lp_text": merged_text,
            "source_row_ids": row_ids,
            "decision_source": "merge",
            "merge_group_id": gid,
            "evidence_para_ids": evid,
            "evidence_text": evidence_text,
        }
    )
    merged_done.update(row_ids)

print("Merge groups applied. Final rows so far:", len(final_rows))

# -------------------------
# (F) Apply non-merged rows
# -------------------------
for _, r in df.iterrows():
    row_id = str(r["row_id"])
    if row_id in merged_done:
        continue

    decision = str(r["decision"]).strip().lower()
    if decision in {"", "reject"}:
        continue

    case_group = str(r["case_group"])
    brief_uid = str(r["brief_uid"])
    brief_role = str(r["brief_role"])
    raw_lp_id = str(r["raw_lp_id"]).strip()

    evid = sorted(
        set(parse_evidence_ids(str(r["evidence_para_ids_joined"]))),
        key=sort_pid,
    )
    evidence_text = join_evidence_text(case_group, brief_uid, evid)

    if decision == "accept":
        text = str(r["lp_text"]).strip()
        tag = "ACCEPT"
    elif decision == "edit":
        text = str(r["edit_lp_text"]).strip() or str(r["lp_text"]).strip()
        tag = "EDIT"
    elif decision == "split":
        children = [c.strip() for c in str(r["split_children"]).split("|||") if c.strip()]
        if not children:
            children = [str(r["lp_text"]).strip()]
        for j, child in enumerate(children, start=1):
            rid_src = (
                f"{case_group}|{brief_uid}|{brief_role}|SPLIT|{j}|"
                f"{child}|{';'.join(evid)}|{row_id}"
            )
            fid = "F-" + sha1(rid_src.encode("utf-8")).hexdigest()[:10]
            final_rows.append(
                {
                    "case_group": case_group,
                    "brief_uid": brief_uid,
                    "brief_role": brief_role,
                    "final_lp_id": fid,
                    "lp_text": child,
                    "source_row_ids": [row_id],
                    "decision_source": "split",
                    "merge_group_id": "",
                    "evidence_para_ids": evid,
                    "evidence_text": evidence_text,
                }
            )
        continue
    else:
        print("Warning: unknown decision value:", decision, "| row_id:", row_id)
        continue

    # NEW: for non-merge, non-split rows, set final_lp_id = raw_lp_id (fallback to old hash if missing)
    if raw_lp_id:
        fid = raw_lp_id
    else:
        rid_src = f"{case_group}|{brief_uid}|{brief_role}|{tag}|{text}|{';'.join(evid)}|{row_id}"
        fid = "F-" + sha1(rid_src.encode("utf-8")).hexdigest()[:10]

    final_rows.append(
        {
            "case_group": case_group,
            "brief_uid": brief_uid,
            "brief_role": brief_role,
            "final_lp_id": fid,
            "lp_text": text,
            "source_row_ids": [row_id],
            "decision_source": decision,
            "merge_group_id": "",
            "evidence_para_ids": evid,
            "evidence_text": evidence_text,
        }
    )

# -------------------------
# (G) Write outputs
# -------------------------
with FINAL_JSONL.open("w", encoding="utf-8") as f:
    for r in final_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

pd.DataFrame(final_rows).to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")

print("Final LP rows:", len(final_rows))
print("Cases:", pd.DataFrame(final_rows)["case_group"].nunique() if final_rows else 0)
print("Briefs:", pd.DataFrame(final_rows)["brief_uid"].nunique() if final_rows else 0)
print("Roles:", sorted(pd.DataFrame(final_rows)["brief_role"].unique()) if final_rows else [])
print("Wrote:", FINAL_JSONL)
print("Wrote:", FINAL_CSV)


Using expert-audited CSV: /home/jupyter-hzh308@uky.edu/scotus_stance/input/expert_audited/expert_audited_lps.csv
Decision counts: {'accept': 130, 'reject': 37, 'edit': 5, 'split': 2}
Merge groups applied. Final rows so far: 0
Final LP rows: 138
Cases: 4
Briefs: 9
Roles: ['petitioner', 'respondent']
Wrote: /home/jupyter-hzh308@uky.edu/scotus_stance/output/final/lps_final.jsonl
Wrote: /home/jupyter-hzh308@uky.edu/scotus_stance/output/final/lps_final.csv


## Expert audit protocol (Markdown)

Copy/paste the protocol below into your project docs (e.g., `scotus_stance/docs/expert_audit_protocol.md`).

### Goal
Validate and refine the list of **contestable legal propositions (LPs)** extracted from the **Summary of Argument (SoA)**. Each LP should be a normative, judge-addressable claim that could be supported, opposed, or probed in oral argument.

### Inputs
- `lps_audit.csv` (one row per LP), containing:
  - `lp_text`
  - `evidence_para_ids_joined`
  - `evidence_text` (full paragraph text for each evidence ID)

### Reviewer actions (per LP)
Fill the `decision` column with one of:
- `accept`: LP is correct and well-formed.
- `edit`: LP is substantively correct but wording should change.
- `reject`: LP is not a substantive proposition, is redundant, or is unsupported by evidence.
- `merge`: LP should be merged with other LP(s) expressing the same contestable judgment.
- `split`: LP contains multiple independent inferential steps and must be split into atomic propositions.

### How to complete each decision
#### accept
- Leave `edit_lp_text` empty.
- Optionally add `notes`.

#### edit
- Put the revised LP wording into `edit_lp_text`.
- Keep the proposition *case-grounded* and *contestable* (do not over-neutralize or over-claim).

#### reject
- Add a brief reason in `notes` (e.g., “pure citation”, “not contestable”, “no support in evidence”).

#### merge
- Assign the same `merge_group_id` to all rows that should be merged (e.g., `M01`, `M02`).
- Optionally use `notes` to specify which row is the best canonical wording.
- The pipeline will create one finalized LP per `merge_group_id`.

#### split
- Put child LPs into `split_children` using the delimiter ` ||| `.
  - Example: `A ||| B ||| C`
- Each child should be an *atomic* contestable claim.

### Evidence rule
An LP is supported only if at least one cited paragraph **substantively advances** the proposition (not merely mentions the topic). Add/remove evidence IDs only if you are confident; otherwise note the issue in `notes`.

### Outputs
After review, the pipeline produces `lps_final.jsonl`, containing the finalized LPs with:
- `final_lp_id`, `lp_text`
- provenance (`source_row_ids`, `decision_source`)
- evidence (`evidence_para_ids`, `evidence_text`)
